# Iceberg internals, hands on
Run each cell in order. You will look under the hood of the tables your pipeline built: snapshots, manifests, data files, time travel, schema evolution and partition evolution.

Everything here runs on the local open-source stack: object store (S3 API) + Iceberg REST catalog + Spark.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sql('SHOW NAMESPACES IN lake').show()

## 1. A table is just metadata pointing at files
`files` lists every data file the current snapshot uses. For **bronze.trips** the paths are in `s3://landing/`: the table was registered **in place** (zero copy).

In [ ]:
spark.sql('SELECT file_path, record_count, file_size_in_bytes FROM lake.bronze.trips.files').show(truncate=False)

## 2. Every write is a snapshot
Iceberg never edits files. A write adds new files and a new snapshot; readers see a consistent version.

In [ ]:
spark.sql('SELECT committed_at, snapshot_id, operation, summary["added-records"] AS added FROM lake.silver.trips.snapshots ORDER BY committed_at').show(truncate=False)

## 3. Time travel
Query the table as it was at an older snapshot. Useful for debugging, audits and reproducible ML training sets.

In [ ]:
first = spark.sql('SELECT snapshot_id FROM lake.silver.trips.snapshots ORDER BY committed_at LIMIT 1').first()[0]
print('rows now:', spark.table('lake.silver.trips').count())
print('rows at first snapshot:', spark.sql(f'SELECT count(*) FROM lake.silver.trips VERSION AS OF {first}').first()[0])

## 4. Manifests: how engines skip files
Each manifest stores min/max values per column for every file. A filter like `pickup_ts >= ...` skips files whose range cannot match.

In [ ]:
spark.sql('SELECT path, added_data_files_count, partition_summaries FROM lake.silver.trips.manifests').show(truncate=80)

## 5. Schema evolution (no rewrite)
Add a column. Old files are untouched; readers return NULL for them.

In [ ]:
spark.sql('ALTER TABLE lake.silver_spark.trips ADD COLUMNS (surge_multiplier DOUBLE)')
spark.table('lake.silver_spark.trips').select('trip_id', 'surge_multiplier').show(3)

## 6. Partition evolution (no rewrite)
Change the partitioning from day to month for new data. Old files keep their layout; queries work across both.

In [ ]:
spark.sql('ALTER TABLE lake.silver_spark.trips REPLACE PARTITION FIELD days(pickup_ts) WITH months(pickup_ts)')
spark.sql('SELECT spec_id, count(*) FROM lake.silver_spark.trips.files GROUP BY spec_id').show()

## 7. Maintenance you will be asked about in interviews
- `rewrite_data_files`: compact small files
- `expire_snapshots`: drop old versions (and their unreferenced files)
- `rewrite_manifests`: speed up planning

**Careful with zero-copy tables:** expiring snapshots or dropping with PURGE can delete files you do not own. That is why Bronze has `gc.enabled=false` and Trino may not DROP it.

In [ ]:
spark.sql("CALL lake.system.rewrite_data_files(table => 'silver_spark.trips')").show()
spark.sql("CALL lake.system.expire_snapshots(table => 'silver_spark.trips', retain_last => 2)").show()